In [1]:
import pandas as pd
import numpy as np
import os
from nielsen_shared import (
    get_panel_dir,
    get_hierarchy_upcs,
    get_table,
    transform_brand,
    BASE_DIR
)

pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', 500)

In [2]:
def analyze_year(year):
    print(f"Analyzing year {year}")
    panel_dir = get_panel_dir(year)
    
    df_hierarchy, upcs = get_hierarchy_upcs(year)
    df_attributes = get_table("productattributes.tsv", year, upcs)
    df_attributes["BRAND"] = transform_brand(df_attributes["BRAND"])
    df_desc = get_table("productdesc.tsv", year, upcs)
    df_attributes_expanded = (df_attributes
        .merge(df_hierarchy, on=["upc", "year"], how="left")
        .merge(df_desc, on=["upc", "year"], how="left"))
    df_purchase = get_table("purchase.tsv", year, upcs, panel=True)  
    df_purchase_expanded = df_purchase.merge(df_attributes_expanded, on="upc", how="left")
    
    df_top_brands_price = df_purchase_expanded.groupby(["BRAND", "sub_category"])["total_price_paid"].sum().sort_values(ascending=False).reset_index()
    df_top_brands_props = df_purchase_expanded.groupby(["BRAND", "sub_category"])["quantity"].sum().sort_values(ascending=False).reset_index()
    df_top_brands = df_top_brands_price.merge(df_top_brands_props, on=["BRAND", "sub_category"], how="inner")
    df_top_brands.to_csv(os.path.join(panel_dir, "laundry_top_brands.csv"))
    df_upc_totals = df_purchase_expanded.groupby(["BRAND", "sub_category", "upc"])["total_price_paid"].sum()
    df_upc_totals_expanded = (df_upc_totals
        .reset_index()
        .merge(df_attributes_expanded[["upc", "product_descr"]], on="upc", how="left")
        .sort_values(by=["BRAND", "sub_category", "total_price_paid"], ascending=False))
    df_upc_totals_expanded.to_csv(os.path.join(panel_dir, "laundry_upc_totals.csv"))

In [3]:
analyze_year("2021")
analyze_year("2022")
analyze_year("2023")
analyze_year("2024")

Analyzing year 2021
Analyzing year 2022
Analyzing year 2023
Analyzing year 2024


In [4]:
years = ["2021", "2022", "2023", "2024"]
dfs_top_brands = []
dfs_upc_totals = []
for year in years:
    base_dir = get_panel_dir(year)
    
    df_top_brands_year = pd.read_csv(os.path.join(base_dir, "laundry_top_brands.csv"), index_col=0)
    df_top_brands_year["year"] = year
    dfs_top_brands.append(df_top_brands_year)

    df_upc_totals = pd.read_csv(os.path.join(base_dir, "laundry_upc_totals.csv"), index_col=0)
    df_upc_totals["year"] = year
    dfs_upc_totals.append(df_upc_totals)
    
df_top_brands = pd.concat(dfs_top_brands, axis=0)
df_upc_totals = pd.concat(dfs_upc_totals, axis=0)

In [5]:
total_spend = df_top_brands["total_price_paid"].sum()
total_quantity = df_top_brands["total_price_paid"].sum()
df_brand_subcategory = df_top_brands.groupby(["BRAND", "sub_category"])[["total_price_paid", "quantity"]].sum().reset_index()
df_brand_subcategory["total_price_paid"] = df_brand_subcategory["total_price_paid"] / total_spend
df_brand_subcategory["quantity"] = df_brand_subcategory["quantity"] / total_quantity
df_brand_subcategory = df_brand_subcategory.sort_values(by="total_price_paid", ascending=False).reset_index(drop=True)
df_brand_subcategory["cumulative percent"] = df_brand_subcategory["total_price_paid"].cumsum().apply(lambda x: f"{(x*100):.2f}%")
df_brand_subcategory["percent"] = df_brand_subcategory["total_price_paid"].apply(lambda x: f"{(x*100):.2f}%")
df_brand_subcategory = df_brand_subcategory.drop("quantity", axis=1)
df_brand_subcategory.to_csv(os.path.join(BASE_DIR, "brand_sub_category_full.csv"))
df_brand_subcategory_subset = df_brand_subcategory.iloc[:10]
df_brand_subcategory_subset.to_csv(os.path.join(BASE_DIR, "brand_sub_category.csv"))